In [ ]:
import json
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# Load colors from JSON
with open('colors.json', 'r') as f:
    colors = json.load(f)

In [ ]:
def plot_color_grid(color_list, cols=10):
    rows = (len(color_list) + cols - 1) // cols
    
    # Set up the figure. We make rows slightly taller (1.2) to fit text
    fig, ax = plt.subplots(figsize=(cols, rows * 1.2))
    ax.set_xlim(0, cols)
    ax.set_ylim(0, rows * 1.2)
    ax.axis('off')
    ax.invert_yaxis() # Top-to-bottom layout
    ax.set_aspect('equal') # Enforce exactly 1:1 aspect ratio so squares stay square
    
    for i, color in enumerate(color_list):
        x = i % cols
        y = (i // cols) * 1.2 # Multiply row index by 1.2 for spacing
        
        hex_color = f"#{color['r']:02x}{color['g']:02x}{color['b']:02x}"
        
        # Draw the color swatch as a perfect 0.8 x 0.8 square
        rect = patches.Rectangle((x + 0.1, y + 0.1), 0.9, 0.8, facecolor=hex_color, edgecolor='none')
        ax.add_patch(rect)
        
        # Draw the index number centered below the swatch (larger Y is visually lower)
        ax.text(x + 0.55, y + 1.05, str(i + 1), ha='center', va='center', fontsize=10, color='black')
    
    plt.show()


# Baseline: RGB -> HSV -> sort by Hue

In [ ]:
# Convert RGB to HSV and sort by Hue using OpenCV
def sort_by_hue(color):
    # OpenCV expects a 3D numpy array for color conversion (e.g., shape (1, 1, 3) for a single pixel)
    rgb_pixel = np.uint8([[[color['r'], color['g'], color['b']]]])
    hsv_pixel = cv2.cvtColor(rgb_pixel, cv2.COLOR_RGB2HSV)
    
    # Extract the Hue value (index 0)
    h = hsv_pixel[0][0][0]
    return int(h)

sorted_colors = sorted(colors, key=sort_by_hue)

In [ ]:
# Display colors in a grid
plot_color_grid(sorted_colors)


## Visualize HSV Space and Hue Projection

In [ ]:
import plotly.graph_objects as go
import cv2
import numpy as np

# Get HSV values for all original colors
hsv_colors = []
for c in colors:
    rgb_pixel = np.uint8([[[c['r'], c['g'], c['b']]]])
    hsv_pixel = cv2.cvtColor(rgb_pixel, cv2.COLOR_RGB2HSV)
    hsv_colors.append(hsv_pixel[0][0])
hsv_colors = np.array(hsv_colors)

H = hsv_colors[:, 0]
S = hsv_colors[:, 1]
V = hsv_colors[:, 2]

scatter_colors = [f"#{c['r']:02x}{c['g']:02x}{c['b']:02x}" for c in colors]

fig = go.Figure()

# Scatter plot of colors in HSV space
fig.add_trace(go.Scatter3d(
    x=H, y=S, z=V,
    mode='markers',
    marker=dict(
        size=5,
        color=scatter_colors,
        line=dict(color='black', width=1)
    ),
    name='Colors'
))

# Plot the Hue projection line running through the mean S and V
mean_S = S.mean()
mean_V = V.mean()
min_H = H.min()
max_H = H.max()

fig.add_trace(go.Scatter3d(
    x=[min_H, max_H], y=[mean_S, mean_S], z=[mean_V, mean_V],
    mode='lines',
    line=dict(color='red', width=5),
    name='Hue 1D Projection Line'
))

# Draw dotted lines from each point to its projection on the Hue axis
for i in range(len(hsv_colors)):
    fig.add_trace(go.Scatter3d(
        x=[H[i], H[i]], y=[S[i], mean_S], z=[V[i], mean_V],
        mode='lines',
        line=dict(color='gray', width=2, dash='dot'),
        showlegend=False
    ))

fig.update_layout(
    title='Colors in HSV space with Hue 1D Projection',
    scene=dict(
        xaxis_title='H (Hue)',
        yaxis_title='S (Saturation)',
        zaxis_title='V (Value)'
    ),
    margin=dict(l=0, r=0, b=0, t=40)
)

fig.show()


# Alt Method 1: RGB -> CIELAB -> sort by PCA in 1D

In [ ]:
from sklearn.decomposition import PCA

# Convert RGB to LAB
def rgb_to_lab(color):
    rgb_pixel = np.uint8([[[color['r'], color['g'], color['b']]]])
    lab_pixel = cv2.cvtColor(rgb_pixel, cv2.COLOR_RGB2LAB)
    return lab_pixel[0][0]

# Get all LAB values
lab_colors = np.array([rgb_to_lab(c) for c in colors])

# Apply PCA to find the principal axis
pca = PCA(n_components=1)
pca_values = pca.fit_transform(lab_colors)

# Combine colors with their 1D PCA values and sort
colors_with_pca = list(zip(colors, pca_values.flatten()))
colors_with_pca.sort(key=lambda x: x[1])
pca_sorted_colors = [c[0] for c in colors_with_pca]

In [ ]:
# Display colors in a grid
plot_color_grid(pca_sorted_colors)


## Visualize CIELAB Space and PCA Projection

In [ ]:
import plotly.graph_objects as go

# Extract L, A, B values
L = lab_colors[:, 0]
A = lab_colors[:, 1]
B = lab_colors[:, 2]

# Get hex colors for scatter plot colors
scatter_colors = [f"#{c['r']:02x}{c['g']:02x}{c['b']:02x}" for c in colors]

fig = go.Figure()

# Scatter plot of colors in LAB space
fig.add_trace(go.Scatter3d(
    x=A, y=B, z=L,
    mode='markers',
    marker=dict(
        size=5,
        color=scatter_colors,
        line=dict(color='black', width=1)
    ),
    name='Colors'
))

# Plot the PCA projection line
mean_lab = pca.mean_
component = pca.components_[0]
min_val = pca_values.min()
max_val = pca_values.max()
line_pts = np.array([mean_lab + min_val * component, mean_lab + max_val * component])

fig.add_trace(go.Scatter3d(
    x=line_pts[:, 1], y=line_pts[:, 2], z=line_pts[:, 0],
    mode='lines',
    line=dict(color='red', width=5),
    name='PCA 1D Projection Line'
))

# Draw dotted lines from each point to its projection on the principal axis
for i in range(len(lab_colors)):
    pt = lab_colors[i]
    proj_pt = mean_lab + pca_values[i][0] * component
    fig.add_trace(go.Scatter3d(
        x=[pt[1], proj_pt[1]], y=[pt[2], proj_pt[2]], z=[pt[0], proj_pt[0]],
        mode='lines',
        line=dict(color='gray', width=2, dash='dot'),
        showlegend=False
    ))

fig.update_layout(
    title='Colors in CIELAB space with PCA 1D Projection',
    scene=dict(
        xaxis_title='A* (Green-Red)',
        yaxis_title='B* (Blue-Yellow)',
        zaxis_title='L* (Lightness)'
    ),
    margin=dict(l=0, r=0, b=0, t=40)
)

fig.show()
